# IRT Models

### Dichotomous models
1. Rasch model (1pl)
2. 2pl
3. 3pl

### Polytomous models
1. Graded Response Model (GRM)
2. Partial Credit Model (PCM)
3. Generalized Partial Credit Model (GPCM)
4. Rating Scale Model (RSM)
5. Nominal Response Model (NRM)
6. Sequential Model
7. Multiple-Choice Model (MCM)

### Multi-facet IRT models
1. Many-Facet Rasch Model (MFRM)
2. Multi-Facet Partial Credit Model (MF-PCM)
3. Multi-Facet Rating Scale Model (MF-RSM)
4. Multi-Facet Generalized Partial Credit Model (MF-GPCM)
5. Multi-Facet Graded Response Model (MF-GRM)
6. Hierarchical Rater Model (HRM)
7. Explanatory IRT with facet covariates (e.g., linear logistic test model with facets)

---

## PCM vs. GRM 비교 분석

PCM(Partial Credit Model)과 GRM(Graded Response Model)은 둘 다 **순서형 다분 반응**(예: 0, 1, 2, 3점 채점)을 위한 IRT 모형입니다.  
둘은 응답 확률을 구성하는 방식이 근본적으로 다릅니다.

---

### 1. 수식 비교

| 속성 | GRM | PCM |
|------|-----|-----|
| **확률 계산 방식** | 누적 확률의 차이 | 인접 카테고리 로그 오즈 |
| **파라미터** | $a_i$ (변별도), $b_{ik}$ (경계 난이도) | $\delta_{ik}$ (단계 난이도) |
| **변별도** | 아이템마다 다를 수 있음 | 모든 아이템 = 1 (Rasch 모형) |
| **임계값 순서** | 반드시 $b_{i1} < b_{i2} < \cdots$ | 순서 제약 없음 (disordered 허용) |
| **Stan 타입** | `ordered[K-1]` 강제 | `vector[K-1]` (자유) |

#### GRM
$$P(X_{ji} \geq k \mid \theta_j) = \text{logistic}\bigl(a_i(\theta_j - b_{ik})\bigr)$$
$$P(X_{ji} = k \mid \theta_j) = P(X_{ji} \geq k) - P(X_{ji} \geq k+1)$$

→ 누적 확률들의 **단조 감소** 곡선 $K-1$개를 먼저 정의하고, 그 차이로 카테고리 확률을 계산.

#### PCM
$$\log \frac{P(X_{ji}=k)}{P(X_{ji}=k-1)} = \theta_j - \delta_{ik}$$
$$P(X_{ji}=k) = \frac{\exp(\ell_{ik})}{\sum_{c}\exp(\ell_{ic})}, \quad \ell_{ik} = \sum_{m=1}^{k}(\theta_j - \delta_{im}), \;\ell_{i0}\equiv 0$$

→ 인접 카테고리 간 **로그 오즈**를 직접 모형화하고, Softmax로 정규화.

---

### 2. 핵심 차이: 임계값 순서 제약

- **GRM**: $b_{i1} < b_{i2} < b_{i3}$ 을 반드시 만족해야 누적 확률이 단조 감소.  
  그렇지 않으면 $P(X \geq k) > P(X \geq k-1)$ 이 되어 음수 카테고리 확률이 발생.

- **PCM**: $\delta_{ik}$에 순서 제약이 없음. Disordered thresholds가 발생하면  
  특정 카테고리가 최빈 카테고리가 되는 $\theta$ 구간이 사라짐 ("유령 카테고리").  
  이는 채점 기준 재검토 신호이지, 모형 자체의 오류는 아님.

---

### 3. 어떤 모형을 선택할까?

| 상황 | 추천 모형 |
|------|----------|
| 아이템마다 변별도가 다를 것으로 예상 | **GRM** |
| 모든 아이템이 동일 변별도 (Rasch 관점) | **PCM** |
| 채점 기준이 명확하고 등간적 | **GRM** |
| 채점 기준이 모호하거나 채점자 재량이 큼 | **PCM** |
| 응답 과정이 단계적 (전 단계 통과 후 다음 단계) | Sequential 모형 |
| 아이템 수가 많고 파라미터 수를 줄이고 싶음 | **RSM** (δ = b + τ) |

아래 코드 셀에서 같은 문항을 GRM과 PCM으로 모형화했을 때  
확률 곡선(CRC)이 어떻게 다른지 시각적으로 비교합니다.

---

## Rasch Model: Interpretation of Estimation Results

### Background: From Classical Test Theory (CTT) to IRT

**Classical Test Theory (CTT)** — also called *True Score Theory* — models an observed score $X$ as:

$$X = T + E$$

where $T$ is the **true score** (the expected score over hypothetical replications) and $E$ is **random measurement error** with $\mathbb{E}[E]=0$.  The central reliability index is Cronbach's $\alpha$ (or KR-20 for dichotomous items), which estimates the ratio of true-score variance to observed-score variance:

$$\alpha = \frac{K}{K-1}\left(1 - \frac{\sum_{i=1}^K \sigma^2_{X_i}}{\sigma^2_X}\right)$$

**Limitations of CTT** that motivated IRT:

| CTT Problem | IRT Solution |
|---|---|
| Person ability estimates depend on which items are used (sample-dependent) | Ability $\theta$ is estimated on an invariant logit scale |
| Item statistics (difficulty = proportion correct $p$) depend on the sample | Item parameters are population-invariant under correct model |
| Reliability is a single number for the whole test | IRT provides item- and person-level **standard errors** |
| Cannot predict performance on items not in the test | ICC extrapolates continuously over $\theta$ |
| Equal weighting of all items regardless of quality | High-discrimination items contribute more to $\theta$ estimation |

---

### The Rasch Model

The **Rasch model** (Georg Rasch, 1960) is the simplest IRT model for dichotomous responses:

$$P(X_{ji} = 1 \mid \theta_j, b_i) = \frac{e^{\theta_j - b_i}}{1 + e^{\theta_j - b_i}} = \text{logistic}(\theta_j - b_i)$$

- $\theta_j$ : Person $j$'s ability (latent trait), measured in **logits**
- $b_i$ : Item $i$'s difficulty, measured in **logits**
- When $\theta_j = b_i$, the probability of a correct response is exactly **0.5**

The model has a single parameter per item (difficulty only); discrimination is fixed at $a=1$ for all items.

---

### Key Numbers to Examine in the Estimation Output

#### 1. Item Difficulty ($\hat{b}_i$) — *logit scale*

| Value | Interpretation |
|---|---|
| $\hat{b}_i \approx 0$ | Item is at the center of the ability distribution |
| $\hat{b}_i > 0$ | Harder item (requires above-average ability) |
| $\hat{b}_i < 0$ | Easier item (below-average ability suffices) |
| $|\hat{b}_i| > 3$ | Extreme item — poorly targeted at most examinees |

**What to check:**
- Range of $\hat{b}$ should span the range of $\hat{\theta}$ for good test targeting
- Standard error $\text{SE}(\hat{b}_i)$ should be small (< 0.3 logits for operational tests)
- Recovery: $\text{corr}(\hat{b}, b_{\text{true}}) > 0.95$ indicates good estimation

#### 2. Person Ability ($\hat{\theta}_j$) — *logit scale*

| Value range | Interpretation |
|---|---|
| $\hat{\theta}_j \in [-4, 4]$ | Typical range for a normally distributed population |
| $\hat{\theta}_j > 4$ | Very high ability (few items target this person) |
| $\hat{\theta}_j < -4$ | Very low ability (floor effect risk) |

**What to check:**
- Distribution shape: should approximate Normal if the population is roughly normal
- $\text{SE}(\hat{\theta}_j)$: smaller for persons with moderate ability, larger for extreme scorers
- Person fit statistics (infit/outfit) detect aberrant response patterns

#### 3. Wright Map (Variable Map)

A **Wright map** places $\hat{b}_i$ and $\hat{\theta}_j$ on the **same logit scale**, enabling direct comparison:

```
  Logit | Persons (θ̂)   | Items (b̂)
  ------+----------------+---------------------
   +2   |   ●            |         [Item 7]
   +1   |  ●●●           |  [Item 3] [Item 9]
    0   | ●●●●●●●        |  [Item 1] [Item 5]
   -1   |  ●●●●          |  [Item 2] [Item 8]
   -2   |    ●           |         [Item 4]
```

- **Well-targeted test**: item difficulties span the same range as person abilities
- **Ceiling effect**: many persons above the hardest item
- **Floor effect**: many persons below the easiest item

#### 4. Model Fit Statistics

| Statistic | Formula | Acceptable range |
|---|---|---|
| Item infit (MNSQ) | Weighted mean-square residual | $0.7 – 1.3$ |
| Item outfit (MNSQ) | Unweighted mean-square residual | $0.7 – 1.3$ |
| Person infit | Same, person-level | $0.7 – 1.3$ |
| Item infit (ZSTD) | Standardized infit | $|z| < 2.0$ |

- **Infit > 1.3**: item shows more variance than the model predicts (unexpected responses)
- **Infit < 0.7**: item is too predictable (possible cueing or dependency)
- Outfit is sensitive to extreme scorers and is often more volatile

#### 5. Reliability Indices

| Index | Meaning |
|---|---|
| **Person separation reliability** | $G^2 / (1 + G^2)$, where $G$ is person separation ratio; analogous to Cronbach's $\alpha$ |
| **Person separation index** $G$ | Range of ability / average SE; $G > 2$ means test distinguishes at least 2 levels |
| **Item separation reliability** | Replicability of item difficulty hierarchy across samples; should be $> 0.90$ |

#### 6. Bayesian Estimation (Stan) Output

When using Bayesian MCMC (as in these notebooks), examine:

| Output | What to look for |
|---|---|
| Posterior mean $\hat{\theta}_j$ | Point estimate of person ability |
| Posterior SD $\sigma_{\theta_j}$ | Posterior uncertainty (analogous to SE) |
| $\hat{R}$ (R-hat) | Convergence: $\hat{R} < 1.05$ for all parameters |
| Effective sample size (ESS) | $> 400$ per parameter for stable estimates |
| Posterior predictive check | Simulated data should match observed response patterns |


---

## Historical Relationship: IRT and Rasch Measurement Model

### Timeline

| Year | Event |
|---|---|
| **1904** | Spearman introduces *g* (general intelligence) and the idea of latent traits |
| **1936** | L. L. Thurstone develops item-response functions (ICCs conceptually) |
| **1952** | Frederic Lord publishes *A Theory of Test Scores*, formally defining ICC-based models |
| **1960** | **Georg Rasch** publishes *Probabilistic Models for Some Intelligence and Attainment Tests* — introduces what later became the Rasch model |
| **1968** | Allan Birnbaum formalizes the 1PL, 2PL, and 3PL as a unified IRT family in *Statistical Theories of Mental Test Scores* |
| **1974** | Benjamin Wright and Nargis Panchapakesan publish the BICAL program — the first widely used Rasch estimation software |
| **1977** | Wright & Stone publish *Best Test Design*, establishing Rasch as a measurement framework independent from IRT |
| **1980** | Lord publishes *Applications of Item Response Theory to Practical Testing Problems* — the canonical IRT textbook |
| **1982** | Masters introduces the **Partial Credit Model** (PCM) — a polytomous extension of Rasch |
| **1986** | Muraki develops the Generalized PCM (GPCM); Samejima's GRM gains wide adoption |
| **1990s** | MCMC methods (Gibbs sampling, Metropolis-Hastings) enable fully Bayesian IRT estimation |
| **2000s** | Multidimensional IRT, explanatory IRT (LLTM), and cognitive diagnosis models emerge |
| **2010s** | Stan/NUTS-HMC enables efficient high-dimensional Bayesian IRT; R packages (TAM, mirt, eRm) proliferate |
| **2020s** | Deep learning–IRT hybrids, large-scale educational data mining, and AI-assisted item generation |

---

### Georg Rasch vs. Frederic Lord: Two Paths to the Same Equation

Both Rasch (in Copenhagen) and Lord (at ETS) arrived at **the same logistic ICC** independently, but from radically different philosophical starting points:

**Rasch's path** (measurement-theoretic):
> Rasch was commissioned to analyze Danish military reading tests and was troubled by the fact that the same child's score changed depending on which test was used. He sought a model in which ability and difficulty are **separable** — i.e., an *objective* measurement model. He was deeply influenced by the measurement philosophy of physics: a scale is only meaningful if the unit is person-independent.

**Lord's path** (statistical-psychometric):
> Lord approached item analysis as a problem of **statistical modeling**: fit the best possible probabilistic model to response data, allow multiple parameters per item to maximize fit, and use maximum likelihood estimation. He explicitly accepted that item parameters and person parameters may interact.

The resulting **conceptual schism** persists today:
- The Rasch tradition sees the model as **a criterion for measurement** ("data must fit the model")
- The IRT tradition sees models as **approximations to data** ("choose the model that fits best")

---

### Key Figures

| Person | Contribution |
|---|---|
| **Georg Rasch** (1901–1980) | Danish statistician; Rasch model and measurement philosophy |
| **Benjamin Wright** (1926–2015) | Rasch's primary advocate in education; WINSTEPS/FACETS software |
| **Frederic Lord** (1912–2000) | IRT pioneer at ETS; Lord's Paradox, 2PL/3PL models |
| **Allan Birnbaum** (1923–1976) | Formalized IRT as a statistical family; introduced information functions |
| **Gerhard Fischer** (1939–) | LLTM and explanatory Rasch models |
| **Eiji Muraki** | GPCM; PARSCALE software |
| **Fumiko Samejima** (1934–) | GRM and graded response theory |
| **David Andrich** | Andrich rating scale model (RSM); Rasch measurement philosophy |


---

## Rasch Measurement Model (RMM) vs. IRT: Concepts and Practical Choice

### The Core Philosophical Difference

| Dimension | **Rasch Measurement Model (RMM)** | **IRT (general)** |
|---|---|---|
| **Epistemology** | Prescriptive: model defines what *counts* as measurement | Descriptive: model approximates the data generation process |
| **Model selection** | Fixed: only the Rasch model (or its polytomous extensions) is used | Data-driven: choose the model (1PL, 2PL, 3PL, GRM, …) that fits best |
| **What to do if data don't fit** | Revise or remove items/persons until fit is achieved | Adopt a more complex model (e.g., add discrimination parameter) |
| **Item parameters** | 1 per item (difficulty only); discrimination is a *construct* property | 1–3 per item; discrimination is an *empirical* property to be estimated |
| **Measurement objectivity** | Separability of person and item parameters is the *goal* | Separability may or may not hold; it's a testable assumption |
| **Sufficiency** | Raw score is a sufficient statistic for $\theta$ | Raw score is generally NOT sufficient (especially in 2PL/3PL) |
| **Primary inference** | Interval-scale measurement; construct validity | Probability prediction; model fit |

---

### Specific Technical Contrasts

#### Specific Objectivity (Rasch's Central Concept)
The Rasch model satisfies **specific objectivity**: the comparison of two persons does not depend on which items are used, and the comparison of two items does not depend on which persons are sampled.

$$\log \frac{P(X_{ji}=1)/P(X_{ji}=0)}{P(X_{j'i}=1)/P(X_{j'i}=0)} = \theta_j - \theta_{j'}$$

This log-odds ratio depends *only* on the persons, not on $b_i$. No other IRT model satisfies this property (unless discrimination is constrained to be equal — which is exactly the Rasch constraint).

#### Conditional Maximum Likelihood (CML) — Rasch only
Because the raw score $r_j = \sum_i X_{ji}$ is a **sufficient statistic** for $\theta_j$ in the Rasch model, item parameters can be estimated by conditioning out persons entirely. This means item difficulty estimates are **unaffected by the ability distribution** of the calibration sample — a unique property of the Rasch model.

Other IRT models (2PL, 3PL) require joint or marginal ML and assume a distribution of $\theta$.

---

### When to Choose RMM vs. IRT

#### Choose **Rasch Measurement Model** when:
- The goal is **measurement** (creating an interval scale comparable across groups, times, or test forms)
- Test development: items are **selected or revised** to achieve measurement quality
- High-stakes certification, licensure, or educational accountability contexts
- Computer-adaptive testing (CAT) frameworks where item banks must be on a common scale
- Small samples (CML is robust; 2PL/3PL require $N \geq 200{-}500$)
- You need **direct score meaning**: a 1-logit difference in $\theta$ means the same thing regardless of which items were used

#### Choose **IRT (2PL, 3PL, GRM, …)** when:
- The goal is **prediction** or **model fit** (understanding item behavior, estimating guessing)
- Items have meaningfully **different discriminations** and forcing equality degrades model fit substantially
- Criterion-referenced score interpretation is primary (not scale development)
- Large samples are available ($N > 500$)
- Exploratory analysis of test data: let the data reveal item characteristics
- Research contexts where model complexity is justified by theoretical considerations

---

### A Practical Decision Framework

```
Is measurement (invariance, interval scale) the primary goal?
    │
    ├── YES → Try Rasch model first
    │         → Examine fit (infit/outfit MNSQ)
    │         → If items misfit: revise or remove
    │         → Accept Rasch if fit is adequate
    │
    └── NO  → Is guessing a serious concern (MCQ)?
                  ├── YES → Consider 3PL or MCM
                  └── NO  → Do items differ substantially in discrimination?
                                ├── YES → 2PL, GPCM, or GRM
                                └── NO  → Rasch / 1PL is parsimonious
```

---

### The Rasch "Sufficient Statistic" Insight in Practice

The practical implication of Rasch's sufficiency property is profound for educators and test developers:

- **Number-correct score is a perfect summary**: In the Rasch model, all information about a person's ability is captured by their total correct score (given items are calibrated). No pattern of right/wrong answers beyond the total matters.
- **Item difficulty is the only meaningful item property**: If an item discriminates differently from others, it is producing a *different* trait, not measuring the same trait better. From the RMM view, high discrimination does not mean "better item" — it may mean the item is pulling on a secondary dimension.
- **Scale construction as hypothesis testing**: Rasch analysis is a test of whether a set of items forms a *unidimensional interval scale*. Failing Rasch fit is not a model failure — it is an empirical finding about the items.


---

## Recent Research Topics in RMM and IRT

### 1. Bayesian IRT and MCMC Estimation
Traditional IRT relies on marginal maximum likelihood (MML) via the EM algorithm. Modern Bayesian approaches using **Stan (NUTS-HMC)**, JAGS, or PyMC enable:
- Full posterior distributions over all parameters (not just point estimates)
- Hierarchical models for item parameters (partial pooling across item types)
- Natural propagation of uncertainty in downstream analyses
- Model comparison via WAIC/LOO-CV instead of $\chi^2$ tests

*Key references*: Fox (2010); Gelman et al. (2013); Burkner (2021) — `brms` + IRT

---

### 2. Explanatory IRT and Cognitive Diagnostic Models (CDM)
The **Linear Logistic Test Model (LLTM)** decomposes item difficulty into cognitive operations:

$$b_i = \sum_p q_{ip} \eta_p + \varepsilon_i$$

where $q_{ip}$ is the Q-matrix weight and $\eta_p$ is the difficulty of cognitive component $p$.

Extensions include:
- **DINA/DINO models**: binary mastery of $K$ skills; $2^K$ latent classes
- **Attribute hierarchy models** (AHM): ordered skill prerequisites
- **General diagnostic models** (GDM, de la Torre 2011)

*Active research*: Q-matrix validation, CDM with continuous skills, CDM-IRT hybrids

---

### 3. Multidimensional IRT (MIRT)
When a test measures multiple correlated traits, unidimensional IRT is misspecified. MIRT models:

$$P(X_{ji}=1 \mid \boldsymbol{\theta}_j, \mathbf{a}_i, d_i) = g\left(\mathbf{a}_i^\top \boldsymbol{\theta}_j + d_i\right)$$

Current research topics:
- **Bifactor models**: general + specific dimensions
- **Exploratory MIRT**: data-driven dimensionality detection (analogous to EFA)
- **Rotation methods** for MIRT solutions (oblique, Procrustes)
- Large-$K$ estimation with variational Bayes or stochastic EM

---

### 4. Computerized Adaptive Testing (CAT) and Item Selection
Rasch-based CAT selects items at the current $\hat{\theta}$ estimate (maximum information = item closest to current ability). IRT-based CAT can use:
- **Maximum Fisher information** (2PL/3PL)
- **Kullback-Leibler information** (polytomous models)
- **Posterior-weighted information** (Bayesian CAT)
- **Constrained CAT**: content balancing, item exposure control, enemy-item constraints

*Hot topic*: Shadow testing (van der Linden & Reese 1998), CAT with multidimensional IRT, and **AI-generated item banks**.

---

### 5. Machine Learning and Deep Learning Extensions
Recent work has proposed hybrid models that relax parametric IRT assumptions:

- **Neural IRT** (NeuralCDM, DIRT): replace the logistic ICC with a neural network
- **Variational autoencoders for IRT** (VAIRT): learn latent ability from response matrices
- **Knowledge tracing** (BKT, DKT, SAINT+): track skill acquisition over time in EdTech platforms
- **Large language model scoring**: automated constructed-response scoring as an IRT-compatible process

*Criticism*: Neural models sacrifice interpretability and measurement-theoretic properties for predictive accuracy.

---

### 6. Fairness, Differential Item Functioning (DIF), and Equity
**DIF** occurs when an item has different ICCs for different groups (e.g., gender, ethnicity) after matching on ability:

$$\text{DIF: } P(X_{ji}=1 \mid \theta, \text{group}=G_1) \neq P(X_{ji}=1 \mid \theta, \text{group}=G_2)$$

Current research:
- **Bayesian DIF detection**: posterior probabilities rather than $p$-value cutoffs
- **Alignment methods**: scale linking when DIF is pervasive (Muthén & Asparouhov, 2014)
- **Intersectional DIF**: multiple group membership simultaneously
- **Algorithmic fairness** in CAT and AI-based scoring

---

### 7. Longitudinal and Growth IRT
Modeling change in latent traits over time:
- **Longitudinal Rasch models**: fixed anchor items across waves
- **Latent change score IRT**: structural equation models with IRT measurement
- **IRT with time-varying covariates**
- **Mixture IRT**: latent class trajectories of ability change

---

### 8. Polytomous Model Comparisons and Selection
With polytomous data (Likert scales, partial credit), model selection among PCM, RSM, GPCM, GRM, Sequential, and NRM is non-trivial:
- **Bayesian model comparison** (WAIC, LOO): preferred over likelihood-ratio tests
- **Ordered Dirichlet priors** for threshold parameters
- **Threshold ordering constraints** (HRM approach) vs. free thresholds (PCM)
- **Within-item factor models** vs. between-item CFA as alternatives to polytomous IRT

---

### 9. Many-Facet Rasch Measurement (MFRM) in Performance Assessment
Extending the Rasch model to rater-mediated assessments:
- **Rater drift detection**: changes in rater severity over a rating session
- **Rater training** using Rasch-based calibration reports
- **Automatic essay scoring** validation against MFRM benchmarks
- **Crossed vs. nested designs**: computational challenges in large-scale MFRM

---

### 10. Software and Open Science
- **R packages**: `TAM`, `mirt`, `eRm`, `ltm`, `sirt`, `brms` (Bayesian)
- **Python**: `girth`, `py-irt`, Stan via `cmdstanpy`
- **Open item banks**: NAEP, PISA data for secondary analysis
- **Reproducibility**: pre-registration of IRT analyses in educational measurement


---

## References

### Foundational Works

**Rasch, G.** (1960). *Probabilistic models for some intelligence and attainment tests*. Danish Institute for Educational Research. (Expanded edition: University of Chicago Press, 1980.)

**Lord, F. M.** (1980). *Applications of item response theory to practical testing problems*. Erlbaum.

**Lord, F. M., & Novick, M. R.** (1968). *Statistical theories of mental test scores*. Addison-Wesley.

**Birnbaum, A.** (1968). Some latent trait models and their use in inferring an examinee's ability. In F. M. Lord & M. R. Novick (Eds.), *Statistical theories of mental test scores* (pp. 397–479). Addison-Wesley.

---

### Rasch Measurement Model

**Wright, B. D., & Stone, M. H.** (1979). *Best test design: Rasch measurement*. MESA Press.

**Wright, B. D., & Masters, G. N.** (1982). *Rating scale analysis: Rasch measurement*. MESA Press.

**Andrich, D.** (1978). A rating formulation for ordered response categories. *Psychometrika, 43*(4), 561–573.

**Masters, G. N.** (1982). A Rasch model for partial credit scoring. *Psychometrika, 47*(2), 149–174.

**Fischer, G. H.** (1973). The linear logistic test model as an instrument in educational research. *Acta Psychologica, 37*(6), 359–374.

**Fischer, G. H., & Molenaar, I. W.** (Eds.). (1995). *Rasch models: Foundations, recent developments, and applications*. Springer.

**Linacre, J. M.** (1989). *Many-facet Rasch measurement*. MESA Press.

**Bond, T. G., & Fox, C. M.** (2015). *Applying the Rasch model: Fundamental measurement in the human sciences* (3rd ed.). Routledge.

**Engelhard, G.** (2013). *Invariant measurement: Using Rasch models in the social, behavioral, and health sciences*. Routledge.

---

### IRT — General Texts

**Hambleton, R. K., Swaminathan, H., & Rogers, H. J.** (1991). *Fundamentals of item response theory*. Sage.

**de Ayala, R. J.** (2009). *The theory and practice of item response theory*. Guilford Press.

**van der Linden, W. J., & Hambleton, R. K.** (Eds.). (1997). *Handbook of modern item response theory*. Springer.

**van der Linden, W. J.** (Ed.). (2016). *Handbook of item response theory* (Vols. 1–3). CRC Press.

**Embretson, S. E., & Reise, S. P.** (2000). *Item response theory for psychologists*. Erlbaum.

---

### Polytomous IRT Models

**Samejima, F.** (1969). Estimation of latent ability using a response pattern of graded scores. *Psychometrika Monograph Supplement, 34*(17).

**Muraki, E.** (1992). A generalized partial credit model: Application of an EM algorithm. *Applied Psychological Measurement, 16*(2), 159–176.

**Thissen, D., & Steinberg, L.** (1986). A taxonomy of item response models. *Psychometrika, 51*(4), 567–577.

**Bock, R. D.** (1972). Estimating item parameters and latent ability when responses are scored in two or more nominal categories. *Psychometrika, 37*(1), 29–51.

---

### Bayesian IRT

**Fox, J.-P.** (2010). *Bayesian item response modeling: Theory and applications*. Springer.

**Albert, J. H.** (1992). Bayesian estimation of normal ogive item response curves using Gibbs sampling. *Journal of Educational Statistics, 17*(3), 251–269.

**Burkner, P.-C.** (2021). Bayesian item response modeling in R with brms and Stan. *Journal of Statistical Software, 100*(5), 1–54.

**Stan Development Team** (2024). *Stan modeling language users guide and reference manual* (Version 2.34). https://mc-stan.org

---

### Modern Extensions and Research

**de la Torre, J.** (2011). The generalized DINA model framework. *Psychometrika, 76*(2), 179–199.

**Reckase, M. D.** (2009). *Multidimensional item response theory*. Springer.

**Muthén, B., & Asparouhov, T.** (2014). IRT studies of many groups: The alignment method. *Frontiers in Psychology, 5*, 978.

**van der Linden, W. J., & Reese, L. M.** (1998). A model for optimal constrained adaptive testing. *Applied Psychological Measurement, 22*(3), 259–270.

**Ames, A. J., & Penfield, R. D.** (2015). An NCME instructional module on item-fit statistics for item response theory models. *Educational Measurement: Issues and Practice, 34*(3), 39–48.

**Liu, Y., Tian, W., & Xin, T.** (2016). An application of M2 statistic to evaluate the fit of cognitive diagnostic models. *Journal of Educational and Behavioral Statistics, 41*(1), 3–26.

**Wang, F., et al.** (2020). Neural cognitive diagnosis for intelligent education systems. *AAAI Conference on Artificial Intelligence, 34*(4), 6153–6161.

---

### Software

**Chalmers, R. P.** (2012). mirt: A multidimensional item response theory package for the R environment. *Journal of Statistical Software, 48*(6), 1–29.

**Kiefer, T., Mayer, A., & Zeileis, A.** (2013). Extended Rasch models in R: The eRm package. *Journal of Statistical Software, 20*(9), 1–20.

**Robitzsch, A., Kiefer, T., & Wu, M.** (2020). *TAM: Test analysis modules*. R package version 3.5-19. https://CRAN.R-project.org/package=TAM

**Linacre, J. M.** (2024). *WINSTEPS Rasch measurement computer program* (Version 5.7). Winsteps.com.

**Linacre, J. M.** (2024). *FACETS Rasch measurement computer program* (Version 3.84). Winsteps.com.


---

## (한국어) Rasch 모델: 추정 결과 해석 가이드

### 배경: 고전검사이론(CTT)에서 문항반응이론(IRT)으로

**고전검사이론(Classical Test Theory, CTT)** — 진점수이론(True Score Theory)이라고도 함 — 은 관찰 점수 $X$를 다음과 같이 모형화합니다:

$$X = T + E$$

여기서 $T$는 **진점수(true score)** (가상의 반복 측정에서 기대되는 점수)이고, $E$는 $\mathbb{E}[E]=0$인 **무선 측정오차**입니다. 대표적인 신뢰도 지수는 Cronbach의 $\alpha$ (이분문항의 경우 KR-20)로, 진점수 분산이 관찰 점수 분산에서 차지하는 비율을 추정합니다:

$$\alpha = \frac{K}{K-1}\left(1 - \frac{\sum_{i=1}^K \sigma^2_{X_i}}{\sigma^2_X}\right)$$

**CTT의 한계와 IRT의 해결책:**

| CTT의 문제점 | IRT의 해결책 |
|---|---|
| 능력 추정치가 사용된 문항에 의존함 (표본 의존적) | 능력 $\theta$는 불변(invariant) 로짓 척도로 추정됨 |
| 문항 통계 (난이도 = 정답률 $p$)가 피험자 표본에 의존함 | 올바른 모형 하에서 문항 모수는 집단 불변 |
| 검사 전체에 대한 단일 신뢰도 계수 | 문항·피험자 수준의 **표준오차** 제공 |
| 검사에 없는 문항에 대한 수행 예측 불가 | ICC가 $\theta$ 전 범위에 걸쳐 연속적으로 확률 예측 |
| 모든 문항을 동등하게 가중 | 변별도가 높은 문항이 $\theta$ 추정에 더 기여 |

---

### Rasch 모델

**Rasch 모델** (Georg Rasch, 1960)은 이분반응에 대한 가장 단순한 IRT 모형입니다:

$$P(X_{ji} = 1 \mid \theta_j, b_i) = \frac{e^{\theta_j - b_i}}{1 + e^{\theta_j - b_i}} = \text{logistic}(\theta_j - b_i)$$

- $\theta_j$ : 피험자 $j$의 능력 (잠재 특성), **로짓(logit)** 단위로 측정
- $b_i$ : 문항 $i$의 난이도, **로짓** 단위로 측정
- $\theta_j = b_i$일 때, 정답 확률은 정확히 **0.5**
- 문항당 모수는 난이도 하나뿐이며, 변별도는 모든 문항에서 $a=1$로 고정

---

### 추정 결과에서 확인해야 할 주요 수치

#### 1. 문항 난이도 ($\hat{b}_i$) — *로짓 척도*

| 값 | 해석 |
|---|---|
| $\hat{b}_i \approx 0$ | 피험자 능력 분포의 중심에 위치하는 문항 |
| $\hat{b}_i > 0$ | 어려운 문항 (평균 이상의 능력 필요) |
| $\hat{b}_i < 0$ | 쉬운 문항 (평균 이하의 능력으로 정답 가능) |
| $|\hat{b}_i| > 3$ | 극단적 문항 — 대부분의 피험자에게 잘 맞지 않음 |

**확인 사항:**
- $\hat{b}$의 범위가 $\hat{\theta}$의 범위를 포괄해야 검사 타게팅(targeting)이 적절함
- 표준오차 $\text{SE}(\hat{b}_i)$는 작을수록 좋음 (실용 검사에서는 0.3 로짓 이하)
- 회수율 검증: $\text{corr}(\hat{b}, b_{\text{true}}) > 0.95$이면 추정이 정확함

#### 2. 피험자 능력 ($\hat{\theta}_j$) — *로짓 척도*

| 값 범위 | 해석 |
|---|---|
| $\hat{\theta}_j \in [-4, 4]$ | 정규분포 모집단에서의 전형적 범위 |
| $\hat{\theta}_j > 4$ | 매우 높은 능력 (이 피험자에 맞는 문항이 거의 없음) |
| $\hat{\theta}_j < -4$ | 매우 낮은 능력 (바닥 효과 위험) |

**확인 사항:**
- 분포 형태: 모집단이 대략 정규분포라면 $\hat{\theta}$도 정규분포에 가까워야 함
- $\text{SE}(\hat{\theta}_j)$: 중간 능력자는 작고, 극단 점수자는 큼
- 피험자 적합도 통계(infit/outfit)로 이상 반응 패턴 탐지

#### 3. Wright Map (변인 지도)

**Wright Map**은 $\hat{b}_i$와 $\hat{\theta}_j$를 **같은 로짓 척도** 위에 배치하여 직접 비교를 가능하게 합니다:

```
  로짓  | 피험자 (θ̂)     | 문항 (b̂)
  ------+----------------+---------------------
   +2   |   ●            |         [문항 7]
   +1   |  ●●●           |  [문항 3] [문항 9]
    0   | ●●●●●●●        |  [문항 1] [문항 5]
   -1   |  ●●●●          |  [문항 2] [문항 8]
   -2   |    ●           |         [문항 4]
```

- **타게팅이 잘 된 검사**: 문항 난이도가 피험자 능력 범위를 포괄
- **천장 효과**: 많은 피험자가 가장 어려운 문항보다 능력이 높음
- **바닥 효과**: 많은 피험자가 가장 쉬운 문항보다 능력이 낮음

#### 4. 모형 적합도 통계량

| 통계량 | 계산 | 허용 범위 |
|---|---|---|
| 문항 infit (MNSQ) | 가중 평균제곱 잔차 | $0.7 – 1.3$ |
| 문항 outfit (MNSQ) | 비가중 평균제곱 잔차 | $0.7 – 1.3$ |
| 피험자 infit | 동일, 피험자 수준 | $0.7 – 1.3$ |
| 문항 infit (ZSTD) | 표준화된 infit | $|z| < 2.0$ |

- **Infit > 1.3**: 모형이 예측하는 것보다 분산이 큼 (예상 밖의 반응 패턴)
- **Infit < 0.7**: 반응이 너무 예측 가능함 (단서 노출, 문항 종속 가능성)
- Outfit은 극단 점수자에 민감하여 변동이 큰 편

#### 5. 신뢰도 지수

| 지수 | 의미 |
|---|---|
| **피험자 분리 신뢰도** | $G^2 / (1 + G^2)$, $G$는 피험자 분리 비율; Cronbach $\alpha$와 유사 |
| **피험자 분리 지수** $G$ | 능력 범위 / 평균 SE; $G > 2$이면 검사가 최소 2개 수준 구분 가능 |
| **문항 분리 신뢰도** | 표본에 걸친 문항 난이도 서열의 재현성; $> 0.90$ 권장 |

#### 6. Bayesian 추정(Stan) 출력 확인

이 노트북들처럼 Bayesian MCMC를 사용할 경우 다음을 확인하세요:

| 출력 | 확인 사항 |
|---|---|
| 사후 평균 $\hat{\theta}_j$ | 피험자 능력의 점 추정치 |
| 사후 표준편차 $\sigma_{\theta_j}$ | 사후 불확실성 (SE에 해당) |
| $\hat{R}$ (R-hat) | 수렴 진단: 모든 모수에 대해 $\hat{R} < 1.05$ |
| 유효 표본 크기 (ESS) | 안정적 추정을 위해 모수당 $> 400$ |
| 사후 예측 점검 | 시뮬레이션 데이터가 관찰 반응 패턴과 일치해야 함 |


---

## (한국어) IRT와 Rasch 측정 모형의 역사적 관계

### 연대표

| 연도 | 사건 |
|---|---|
| **1904년** | Spearman이 일반 지능 *g*와 잠재 특성 개념 도입 |
| **1936년** | Thurstone이 문항반응함수(ICC) 개념을 이론적으로 발전시킴 |
| **1952년** | Frederic Lord가 *A Theory of Test Scores*를 발표하며 ICC 기반 모형을 공식화 |
| **1960년** | **Georg Rasch**가 *Probabilistic Models for Some Intelligence and Attainment Tests* 출판 — Rasch 모형의 기원 |
| **1968년** | Allan Birnbaum이 1PL·2PL·3PL을 통합된 IRT 체계로 정식화; 검사 정보 함수 도입 |
| **1974년** | Wright & Panchapakesan이 BICAL 소프트웨어 발표 — 최초의 널리 쓰인 Rasch 추정 프로그램 |
| **1977년** | Wright & Stone이 *Best Test Design* 출판 — Rasch를 IRT와 독립적인 측정 이론으로 정립 |
| **1980년** | Lord가 *Applications of Item Response Theory to Practical Testing Problems* 출판 — IRT 교과서의 표준 |
| **1982년** | Masters가 **부분점수모형(PCM)** 발표 — Rasch의 다분 확장 모형 |
| **1986년** | Muraki가 GPCM 개발; Samejima의 GRM이 널리 수용됨 |
| **1990년대** | MCMC 방법(깁스 표집, Metropolis-Hastings)으로 완전 Bayesian IRT 추정 가능 |
| **2000년대** | 다차원 IRT, 설명적 IRT(LLTM), 인지 진단 모형 등장 |
| **2010년대** | Stan/NUTS-HMC로 고차원 Bayesian IRT 효율화; R 패키지(TAM, mirt, eRm) 확산 |
| **2020년대** | 딥러닝-IRT 혼합 모형, 대규모 교육 데이터 마이닝, AI 보조 문항 생성 |

---

### Georg Rasch vs. Frederic Lord: 같은 공식에 이른 두 갈래 길

Rasch(코펜하겐)와 Lord(ETS)는 **같은 로지스틱 ICC**에 독립적으로 도달했지만, 출발점의 철학이 완전히 달랐습니다.

**Rasch의 경로** (측정이론적):
> Rasch는 덴마크 군 독해력 검사를 분석하는 용역을 맡았다가, 동일한 아동의 점수가 어떤 검사를 쓰느냐에 따라 달라지는 문제를 발견했습니다. 그는 능력과 난이도가 **분리 가능한** — 즉, *객관적* 측정 모형을 추구했습니다. 물리학의 측정 철학에서 깊은 영감을 받았으며, "단위가 사람에 따라 달라지는 척도는 의미가 없다"고 보았습니다.

**Lord의 경로** (통계·심리측정적):
> Lord는 문항 분석을 **통계적 모형화** 문제로 접근했습니다: 반응 데이터에 가장 잘 맞는 확률 모형을 선택하고, 문항당 여러 모수를 허용하여 적합도를 극대화하며, 최대우도추정법을 사용하는 것이 목표였습니다. 문항 모수와 피험자 모수가 상호작용할 수 있음을 명시적으로 받아들였습니다.

이로부터 오늘날까지 이어지는 **개념적 분열**이 생겼습니다:
- Rasch 전통: 모형은 **측정의 기준** ("데이터가 모형에 맞아야 한다")
- IRT 전통: 모형은 **데이터에 대한 근사** ("가장 잘 맞는 모형을 선택하라")

---

### 주요 인물

| 인물 | 기여 |
|---|---|
| **Georg Rasch** (1901–1980) | 덴마크 통계학자; Rasch 모형과 측정 철학 |
| **Benjamin Wright** (1926–2015) | Rasch의 교육학 분야 주창자; WINSTEPS/FACETS 소프트웨어 개발 |
| **Frederic Lord** (1912–2000) | ETS의 IRT 선구자; Lord의 역설, 2PL/3PL 모형 |
| **Allan Birnbaum** (1923–1976) | IRT를 통계 체계로 정식화; 검사 정보 함수 도입 |
| **Gerhard Fischer** (1939–) | LLTM과 설명적 Rasch 모형 |
| **Eiji Muraki** | GPCM; PARSCALE 소프트웨어 |
| **Fumiko Samejima** (1934–) | GRM과 등급반응이론 |
| **David Andrich** | Andrich 평정척도모형(RSM); Rasch 측정 철학 |


---

## (한국어) Rasch 측정 모형(RMM)과 IRT의 개념 비교 및 실용적 선택

### 핵심 철학적 차이

| 차원 | **Rasch 측정 모형 (RMM)** | **IRT (일반)** |
|---|---|---|
| **인식론** | 규범적: 모형이 '측정'의 정의 기준 | 기술적: 모형은 데이터 생성 과정을 근사 |
| **모형 선택** | 고정: Rasch 모형(및 다분 확장)만 사용 | 데이터 주도: 가장 잘 맞는 모형(1PL, 2PL, 3PL, GRM…) 선택 |
| **데이터가 맞지 않으면** | 문항·피험자를 수정 또는 제거하여 적합도 확보 | 더 복잡한 모형(변별도 모수 추가 등) 채택 |
| **문항 모수** | 문항당 1개(난이도만); 변별도는 구인의 속성 | 문항당 1–3개; 변별도는 경험적으로 추정되는 속성 |
| **측정 객관성** | 피험자·문항 모수 분리 가능성이 목표 | 분리 가능성은 검증 가능한 가정 |
| **충분통계량** | 원점수는 $\theta$에 대한 충분통계량 | 원점수는 일반적으로 충분통계량이 아님 (2PL/3PL) |
| **주요 추론 목적** | 등간 척도 측정; 구인 타당도 | 확률 예측; 모형 적합도 |

---

### 구체적 기술 비교

#### 특수 객관성 (Specific Objectivity) — Rasch의 핵심 개념

Rasch 모형은 **특수 객관성**을 만족합니다: 두 피험자의 비교는 어떤 문항을 쓰든 동일하고, 두 문항의 비교는 어떤 피험자 표본을 쓰든 동일합니다.

$$\log \frac{P(X_{ji}=1)/P(X_{ji}=0)}{P(X_{j'i}=1)/P(X_{j'i}=0)} = \theta_j - \theta_{j'}$$

이 로그 오즈 비는 **피험자에만 의존**하며 $b_i$에는 의존하지 않습니다. 변별도를 동일하게 제약하지 않는 한, 다른 어떤 IRT 모형도 이 성질을 만족하지 않습니다.

#### 조건부 최대우도(CML) 추정 — Rasch 모형만의 특성

Rasch 모형에서 원점수 $r_j = \sum_i X_{ji}$는 $\theta_j$에 대한 **충분통계량**이므로, 피험자를 완전히 제거(조건화)하고 문항 모수만을 추정할 수 있습니다. 이는 문항 난이도 추정치가 **피험자 능력 분포에 무관**함을 의미합니다 — Rasch 모형만의 고유한 특성입니다.

2PL, 3PL 등 다른 IRT 모형은 결합 ML 또는 주변 ML을 사용하며 $\theta$의 분포를 가정해야 합니다.

---

### RMM과 IRT 중 무엇을 선택할 것인가

#### **Rasch 측정 모형**을 선택할 때:
- 목표가 **측정** (집단·시간·검사 형식에 걸쳐 비교 가능한 등간 척도 구성)인 경우
- 검사 개발 단계에서 측정 품질을 달성하기 위해 **문항을 선별하거나 수정**하는 경우
- 고부담 자격·면허·교육 책무성 맥락
- 공통 척도 위의 문항 은행이 필요한 컴퓨터 적응 검사(CAT) 체계
- 소표본 (CML은 견고함; 2PL/3PL은 $N \geq 200{-}500$ 필요)
- **점수 의미의 직접 해석**이 필요한 경우: 1로짓 차이는 사용된 문항에 무관하게 동일한 의미

#### **IRT (2PL, 3PL, GRM 등)**를 선택할 때:
- 목표가 **예측** 또는 **모형 적합** (문항 행동 이해, 추측도 추정)인 경우
- 문항 변별도가 실질적으로 달라 동일 제약이 적합도를 크게 낮출 때
- 준거지향 점수 해석이 우선이고 척도 개발이 주목적이 아닌 경우
- 대표본 이용 가능 ($N > 500$)
- 탐색적 검사 데이터 분석: 데이터로 문항 특성을 발견하려는 경우
- 이론적 근거로 모형 복잡성이 정당화되는 연구 맥락

---

### 실용적 의사결정 흐름도

```
측정(불변성, 등간 척도)이 주된 목적인가?
    │
    ├── 예  → Rasch 모형 우선 시도
    │         → 적합도 검토 (infit/outfit MNSQ)
    │         → 부적합 문항: 수정 또는 제거
    │         → 적합도 양호하면 Rasch 채택
    │
    └── 아니오 → 추측(guessing)이 심각한 문제인가? (선다형)
                    ├── 예 → 3PL 또는 MCM 고려
                    └── 아니오 → 문항 변별도 차이가 큰가?
                                    ├── 예 → 2PL, GPCM, 또는 GRM
                                    └── 아니오 → Rasch / 1PL (간명성 우선)
```

---

### Rasch '충분통계량' 개념의 실제적 함의

Rasch의 충분통계량 성질은 교육자와 검사 개발자에게 실질적으로 중요한 의미를 갖습니다:

- **정답 수는 완전한 요약**: Rasch 모형에서 피험자 능력에 관한 모든 정보는 총 정답 수에 담겨 있습니다 (문항이 캘리브레이션된 경우). 총점 외에 어떤 반응 패턴도 추가 정보를 제공하지 않습니다.
- **문항 난이도만이 의미 있는 문항 특성**: 어떤 문항이 다른 문항들과 다른 변별도를 보인다면, 그것은 같은 특성을 '더 잘' 측정하는 것이 아니라 *다른 특성*을 측정하고 있다는 신호입니다. RMM 관점에서 높은 변별도는 "좋은 문항"이 아니라 2차원 문항의 가능성을 의미합니다.
- **척도 구성은 가설 검정**: Rasch 분석은 일련의 문항들이 **일차원 등간 척도**를 이루는지 검정하는 과정입니다. Rasch 적합도 실패는 모형의 실패가 아니라 문항에 대한 경험적 발견입니다.


---

## (한국어) RMM과 IRT의 최신 연구 주제

### 1. Bayesian IRT와 MCMC 추정
전통적 IRT는 EM 알고리즘을 이용한 주변 최대우도(MML) 추정에 의존합니다. **Stan(NUTS-HMC)**, JAGS, PyMC 등을 이용한 현대적 Bayesian 접근법은 다음을 가능하게 합니다:
- 모든 모수에 대한 완전한 사후분포 (점 추정치만이 아닌)
- 문항 모수의 위계적 모형 (문항 유형 간 부분 풀링)
- 하위 분석에서의 자연스러운 불확실성 전파
- $\chi^2$ 검정 대신 WAIC/LOO-CV를 통한 모형 비교

*주요 참고문헌*: Fox (2010); Gelman et al. (2013); Bürkner (2021) — `brms` + IRT

---

### 2. 설명적 IRT와 인지 진단 모형(CDM)
**선형 로지스틱 검사 모형(LLTM)**은 문항 난이도를 인지 조작으로 분해합니다:

$$b_i = \sum_p q_{ip} \eta_p + \varepsilon_i$$

여기서 $q_{ip}$는 Q-행렬 가중치, $\eta_p$는 인지 구성요소 $p$의 난이도입니다.

확장 모형으로는:
- **DINA/DINO 모형**: $K$개 기술의 이분 숙달 여부; $2^K$ 잠재 계층
- **속성 위계 모형(AHM)**: 순서가 있는 선행 기술 구조
- **일반 진단 모형(GDM)** (de la Torre 2011)

*활발한 연구 분야*: Q-행렬 타당화, 연속적 기술을 가진 CDM, CDM-IRT 혼합 모형

---

### 3. 다차원 IRT (MIRT)
검사가 상관된 여러 특성을 측정할 때 일차원 IRT는 모형 오설정이 됩니다. MIRT 모형:

$$P(X_{ji}=1 \mid \boldsymbol{\theta}_j, \mathbf{a}_i, d_i) = g\left(\mathbf{a}_i^\top \boldsymbol{\theta}_j + d_i\right)$$

현재 연구 주제:
- **이분 요인 모형**: 일반 + 특수 차원
- **탐색적 MIRT**: 데이터 주도 차원수 탐지 (탐색적 요인분석과 유사)
- MIRT 해에 대한 **회전 방법** (사교 회전, Procrustes 회전)
- 변분 Bayes 또는 확률적 EM을 이용한 고차원 추정

---

### 4. 컴퓨터 적응 검사(CAT)와 문항 선택
Rasch 기반 CAT는 현재 $\hat{\theta}$ 추정치에서 정보량이 최대인 문항(현재 능력에 가장 가까운 난이도)을 선택합니다. IRT 기반 CAT는 다음을 활용합니다:
- **최대 Fisher 정보** (2PL/3PL)
- **쿨백-라이블러(KL) 정보** (다분 모형)
- **사후 가중 정보** (Bayesian CAT)
- **제약 CAT**: 내용 균형, 문항 노출 통제, 경쟁 문항 제약

*주목받는 주제*: Shadow testing, 다차원 IRT 기반 CAT, **AI 생성 문항 은행**

---

### 5. 기계학습 및 딥러닝 확장 모형
최근 연구들은 IRT의 모수적 가정을 완화하는 혼합 모형을 제안하고 있습니다:

- **Neural IRT** (NeuralCDM, DIRT): 로지스틱 ICC를 신경망으로 대체
- **IRT를 위한 변분 오토인코더** (VAIRT): 반응 행렬에서 잠재 능력 학습
- **지식 추적** (BKT, DKT, SAINT+): EdTech 플랫폼에서 시간에 따른 기술 습득 추적
- **대규모 언어 모형 채점**: 자동 서술형 채점을 IRT 호환 과정으로 처리

*비판적 관점*: 신경 모형은 예측 정확도를 위해 해석 가능성과 측정이론적 특성을 희생합니다.

---

### 6. 공정성, 차별 기능 문항(DIF), 형평성
**DIF**는 능력을 통제한 후 집단(성별, 인종 등)에 따라 문항의 ICC가 다른 경우를 말합니다:

$$\text{DIF: } P(X_{ji}=1 \mid \theta, \text{집단}=G_1) \neq P(X_{ji}=1 \mid \theta, \text{집단}=G_2)$$

현재 연구 주제:
- **Bayesian DIF 탐지**: $p$-값 기준 대신 사후 확률
- **정렬(Alignment) 방법**: DIF가 만연할 때의 척도 연결 (Muthén & Asparouhov, 2014)
- **교차적 DIF**: 복수의 집단 소속을 동시에 고려
- CAT 및 AI 채점에서의 **알고리즘 공정성**

---

### 7. 종단 및 성장 IRT
시간에 따른 잠재 특성 변화 모형화:
- **종단 Rasch 모형**: 측정 시점에 걸쳐 고정 앵커 문항 사용
- **잠재 변화 점수 IRT**: IRT 측정을 결합한 구조방정식 모형
- **시간 가변 공변량을 가진 IRT**
- **혼합 IRT**: 능력 변화의 잠재 계층 궤적

---

### 8. 다분 모형 비교와 선택
다분 데이터(Likert 척도, 부분 점수)에서 PCM, RSM, GPCM, GRM, Sequential, NRM 간의 모형 선택은 쉽지 않습니다:
- **Bayesian 모형 비교** (WAIC, LOO): 우도비 검정보다 선호
- 임계값 모수에 대한 순서 Dirichlet 사전분포
- **임계값 순서 제약** (HRM 방식) vs. 자유 임계값 (PCM)
- 다분 IRT의 대안으로서 문항 내 요인 모형 vs. 문항 간 CFA

---

### 9. 다국면 Rasch 측정(MFRM)과 수행 평가
채점자 매개 평가에서의 Rasch 모형 확장:
- **채점자 표류 탐지**: 채점 회기 내 채점자 엄격성 변화
- Rasch 기반 캘리브레이션 보고서를 활용한 **채점자 훈련**
- **자동 에세이 채점** 타당화 (MFRM 기준 대비)
- 대규모 MFRM에서의 교차(crossed) vs. 중첩(nested) 설계: 계산적 도전

---

### 10. 소프트웨어와 오픈 사이언스
- **R 패키지**: `TAM`, `mirt`, `eRm`, `ltm`, `sirt`, `brms` (Bayesian)
- **Python**: `girth`, `py-irt`, `cmdstanpy`를 통한 Stan
- **공개 문항 은행**: NAEP, PISA 데이터의 2차 분석
- **재현 가능성**: 교육 측정 IRT 분석의 사전 등록(pre-registration)


---

## (한국어) 참고문헌

### 기초 문헌

**Rasch, G.** (1960). *Probabilistic models for some intelligence and attainment tests*. 덴마크 교육연구소. (확장판: University of Chicago Press, 1980.)

**Lord, F. M.** (1980). *Applications of item response theory to practical testing problems*. Erlbaum.

**Lord, F. M., & Novick, M. R.** (1968). *Statistical theories of mental test scores*. Addison-Wesley.

**Birnbaum, A.** (1968). Some latent trait models and their use in inferring an examinee's ability. In F. M. Lord & M. R. Novick (Eds.), *Statistical theories of mental test scores* (pp. 397–479). Addison-Wesley.

---

### Rasch 측정 모형

**Wright, B. D., & Stone, M. H.** (1979). *Best test design: Rasch measurement*. MESA Press.

**Wright, B. D., & Masters, G. N.** (1982). *Rating scale analysis: Rasch measurement*. MESA Press.

**Andrich, D.** (1978). A rating formulation for ordered response categories. *Psychometrika, 43*(4), 561–573.

**Masters, G. N.** (1982). A Rasch model for partial credit scoring. *Psychometrika, 47*(2), 149–174.

**Fischer, G. H.** (1973). The linear logistic test model as an instrument in educational research. *Acta Psychologica, 37*(6), 359–374.

**Fischer, G. H., & Molenaar, I. W.** (Eds.). (1995). *Rasch models: Foundations, recent developments, and applications*. Springer.

**Linacre, J. M.** (1989). *Many-facet Rasch measurement*. MESA Press.

**Bond, T. G., & Fox, C. M.** (2015). *Applying the Rasch model: Fundamental measurement in the human sciences* (3rd ed.). Routledge.

**Engelhard, G.** (2013). *Invariant measurement: Using Rasch models in the social, behavioral, and health sciences*. Routledge.

---

### IRT — 일반 교재

**Hambleton, R. K., Swaminathan, H., & Rogers, H. J.** (1991). *Fundamentals of item response theory*. Sage.

**de Ayala, R. J.** (2009). *The theory and practice of item response theory*. Guilford Press.

**van der Linden, W. J., & Hambleton, R. K.** (Eds.). (1997). *Handbook of modern item response theory*. Springer.

**van der Linden, W. J.** (Ed.). (2016). *Handbook of item response theory* (Vols. 1–3). CRC Press.

**Embretson, S. E., & Reise, S. P.** (2000). *Item response theory for psychologists*. Erlbaum.

---

### 다분 IRT 모형

**Samejima, F.** (1969). Estimation of latent ability using a response pattern of graded scores. *Psychometrika Monograph Supplement, 34*(17).

**Muraki, E.** (1992). A generalized partial credit model: Application of an EM algorithm. *Applied Psychological Measurement, 16*(2), 159–176.

**Thissen, D., & Steinberg, L.** (1986). A taxonomy of item response models. *Psychometrika, 51*(4), 567–577.

**Bock, R. D.** (1972). Estimating item parameters and latent ability when responses are scored in two or more nominal categories. *Psychometrika, 37*(1), 29–51.

---

### Bayesian IRT

**Fox, J.-P.** (2010). *Bayesian item response modeling: Theory and applications*. Springer.

**Albert, J. H.** (1992). Bayesian estimation of normal ogive item response curves using Gibbs sampling. *Journal of Educational Statistics, 17*(3), 251–269.

**Bürkner, P.-C.** (2021). Bayesian item response modeling in R with brms and Stan. *Journal of Statistical Software, 100*(5), 1–54.

**Stan Development Team** (2024). *Stan modeling language users guide and reference manual* (Version 2.34). https://mc-stan.org

---

### 현대적 확장과 연구

**de la Torre, J.** (2011). The generalized DINA model framework. *Psychometrika, 76*(2), 179–199.

**Reckase, M. D.** (2009). *Multidimensional item response theory*. Springer.

**Muthén, B., & Asparouhov, T.** (2014). IRT studies of many groups: The alignment method. *Frontiers in Psychology, 5*, 978.

**van der Linden, W. J., & Reese, L. M.** (1998). A model for optimal constrained adaptive testing. *Applied Psychological Measurement, 22*(3), 259–270.

**Ames, A. J., & Penfield, R. D.** (2015). An NCME instructional module on item-fit statistics for item response theory models. *Educational Measurement: Issues and Practice, 34*(3), 39–48.

**Wang, F., et al.** (2020). Neural cognitive diagnosis for intelligent education systems. *AAAI Conference on Artificial Intelligence, 34*(4), 6153–6161.

---

### 소프트웨어

**Chalmers, R. P.** (2012). mirt: A multidimensional item response theory package for the R environment. *Journal of Statistical Software, 48*(6), 1–29.

**Robitzsch, A., Kiefer, T., & Wu, M.** (2020). *TAM: Test analysis modules*. R package version 3.5-19. https://CRAN.R-project.org/package=TAM

**Linacre, J. M.** (2024). *WINSTEPS Rasch measurement computer program* (Version 5.7). Winsteps.com.

**Linacre, J. M.** (2024). *FACETS Rasch measurement computer program* (Version 3.84). Winsteps.com.

---

### 한국어 참고자료

**성태제** (2019). *문항반응이론의 이해와 적용* (3판). 교육과학사.

**김성숙, 김양분** (2001). *문항반응이론*. 교육과학사.

**탁진국** (2007). *심리검사: 개발과 평가방법의 이해*. 학지사.

**이순묵** (2000). *공변량구조분석*. 성원사. *(측정 모형 관련 배경)*

**한국교육평가학회** (편). (2004). *교육평가 용어사전*. 학지사.
